Configure DeepEval for retrieval accuracy

In [1]:
 #!pip install -U deepeval langchain_aws

In [ ]:
!pip install deepeval langchain-anthropic datasets

In [ ]:
from deepeval import assert_test
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
import boto3
import json
import os
import anthropic

In [ ]:
from anthropic import HUMAN_PROMPT, AI_PROMPT
from dotenv import load_dotenv
load_dotenv("env.txt")

In [ ]:
os.environ['ANTHROPIC_API_KEY'] = ""
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
import os
from datasets import load_dataset
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from langchain_anthropic import ChatAnthropic
from deepeval.models.base_model import DeepEvalBaseLLM

DeepEval with SQuAD

In [ ]:
# Step 1: Create a custom LLM class for Anthropic
class AnthropicLLM(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model
    
    def load_model(self):
        return self.model
    
    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content
    
    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content
    
    def get_model_name(self):
        return "Anthropic Model"

# Step 2: Set up your Anthropic model
anthropic_model = ChatAnthropic(
    model="claude-3-5-sonnet-20240620",  # Replace with your desired Anthropic model
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    temperature=0.0
)

# Initialize the custom LLM wrapper
anthropic_llm = AnthropicLLM(model=anthropic_model)

# Step 3: Load the SQuAD dataset from Hugging Face
def load_squad_dataset():
    dataset = load_dataset("squad", split="validation")  # Load the validation split of the SQuAD dataset
    return dataset

# Step 4: Convert the SQuAD dataset into LLMTestCase objects
def create_test_cases_from_squad(dataset, max_examples=50):
    test_cases = []
    # Use .select() to limit the number of examples
    limited_dataset = dataset.select(range(max_examples))  # Select the first `max_examples` entries
    for example in limited_dataset:
        question = example["question"]  # The question from the SQuAD dataset
        context = example["context"]  # The context paragraph from the SQuAD dataset
        answer = example["answers"]["text"][0]  # The first answer from the list of possible answers
        
        # Create a test case
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[context]
        )
        test_cases.append(test_case)
    return test_cases

# Step 5: Choose and configure the metric for Answer Relevancy
answer_relevancy_metric = AnswerRelevancyMetric(
    model=anthropic_llm,
    threshold=0.5
)

# Step 6: Define a function to calculate Precision@K, Recall@K, F1 Score, and MRR
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

def evaluate_custom_metrics(retrieved_docs, ground_truths, top_k=10):
    """
    Calculate Precision@K, Recall@K, F1 Score, and MRR.
    """
    relevance = [1 if any(gt.strip().lower() in doc.strip().lower() for gt in ground_truths) else 0 for doc, _ in retrieved_docs[:top_k]]
    
    # Precision@K
    precision_at_k = sum(relevance) / top_k if top_k > 0 else 0
    
    # Recall@K
    total_relevant_in_corpus = max(1, sum(1 for gt in ground_truths if any(gt.strip().lower() in doc.strip().lower() for doc, _ in retrieved_docs)))
    recall_at_k = min(sum(relevance) / total_relevant_in_corpus, 1.0) if total_relevant_in_corpus > 0 else 0
    
    # F1 Score
    f1_score_value = 2 * precision_at_k * recall_at_k / (precision_at_k + recall_at_k) if (precision_at_k + recall_at_k) > 0 else 0
    
    # Mean Reciprocal Rank (MRR)
    mrr = 0
    if 1 in relevance:
        mrr = 1 / (relevance.index(1) + 1)
    
    return {
        "Precision@K": precision_at_k,
        "Recall@K": recall_at_k,
        "F1 Score": f1_score_value,
        "MRR": mrr
    }

# Step 7: Run the evaluation
if __name__ == "__main__":
    # Load the SQuAD dataset
    squad_dataset = load_squad_dataset()
    
    # Create test cases from the SQuAD dataset
    test_cases = create_test_cases_from_squad(squad_dataset, max_examples=50)
    
    # Evaluate Answer Relevancy with DeepEval
    answer_relevancy_results = evaluate(test_cases=test_cases, metrics=[answer_relevancy_metric])
    print("Answer Relevancy Results:", answer_relevancy_results)
    
    # Evaluate custom metrics (Precision@K, Recall@K, F1 Score, MRR)
    custom_metrics = []
    for test_case in test_cases:
        retrieved_docs = [(context, 1.0) for context in test_case.retrieval_context]  # Simulate retrieval
        ground_truths = [test_case.actual_output]  # Ground truth answers
        metrics = evaluate_custom_metrics(retrieved_docs, ground_truths, top_k=10)
        custom_metrics.append(metrics)
    
    # Aggregate results
    avg_precision = np.mean([m["Precision@K"] for m in custom_metrics])
    avg_recall = np.mean([m["Recall@K"] for m in custom_metrics])
    avg_f1 = np.mean([m["F1 Score"] for m in custom_metrics])
    avg_mrr = np.mean([m["MRR"] for m in custom_metrics])
    
    print("Custom Metrics:")
    print(f"Average Precision@K: {avg_precision}")
    print(f"Average Recall@K: {avg_recall}")
    print(f"Average F1 Score: {avg_f1}")
    print(f"Average MRR: {avg_mrr}")

DeepEval with FiQA

In [ ]:
# Step 1: Create a custom LLM class for Anthropic
class AnthropicLLM(DeepEvalBaseLLM):
    def __init__(self, model):
        self.model = model
    
    def load_model(self):
        return self.model
    
    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content
    
    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content
    
    def get_model_name(self):
        return "Anthropic Model"

# Step 2: Set up your Anthropic model
anthropic_model = ChatAnthropic(
    model="claude-3-5-sonnet-20240620",  # Replace with your desired Anthropic model
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    temperature=0.0
)

# Initialize the custom LLM wrapper
anthropic_llm = AnthropicLLM(model=anthropic_model)

# Step 3: Load the FiQA dataset from Hugging Face
def load_fiqa_dataset():
    dataset = load_dataset("explodinggradients/fiqa", split="baseline")  # Load the baseline split of the FiQA dataset
    return dataset

# Step 4: Convert the FiQA dataset into LLMTestCase objects
def create_test_cases_from_fiqa(dataset, max_examples=30):
    test_cases = []
    # Use .select() to limit the number of examples
    limited_dataset = dataset.select(range(max_examples))  # Select the first `max_examples` entries
    for example in limited_dataset:
        question = example["question"]  # The question from the FiQA dataset
        contexts = example["contexts"]  # A list of context passages
        ground_truths = example["ground_truths"]  # A list of ground truth answers
        
        # Use the first ground truth as the expected answer (or combine all if needed)
        answer = ground_truths[0]  # You can modify this to handle multiple ground truths
        
        # Combine all contexts into a single retrieval context
        retrieval_context = [context for context in contexts]
        
        # Create a test case
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=retrieval_context
        )
        test_cases.append(test_case)
    return test_cases

# Step 5: Choose and configure the metric
metric = AnswerRelevancyMetric(
    model=anthropic_llm,
    threshold=0.5
)

# Step 6: Run the evaluation
if __name__ == "__main__":
    # Load the FiQA dataset
    fiqa_dataset = load_fiqa_dataset()
    
    # Create test cases from the FiQA dataset
    test_cases = create_test_cases_from_fiqa(fiqa_dataset, max_examples=30)
    
    # Run the evaluation
    evaluate(test_cases=test_cases, metrics=[metric])
    
    custom_metrics = []
    for test_case in test_cases:
        retrieved_docs = [(context, 1.0) for context in test_case.retrieval_context]  # Simulate retrieval
        ground_truths = [test_case.actual_output]  # Ground truth answers
        metrics = evaluate_custom_metrics(retrieved_docs, ground_truths, top_k=10)
        custom_metrics.append(metrics)
    
    # Aggregate results
    avg_precision = np.mean([m["Precision@K"] for m in custom_metrics])
    avg_recall = np.mean([m["Recall@K"] for m in custom_metrics])
    avg_f1 = np.mean([m["F1 Score"] for m in custom_metrics])
    avg_mrr = np.mean([m["MRR"] for m in custom_metrics])
    
    print("Custom Metrics:")
    print(f"Average Precision@K: {avg_precision}")
    print(f"Average Recall@K: {avg_recall}")
    print(f"Average F1 Score: {avg_f1}")
    print(f"Average MRR: {avg_mrr}")